In [ ]:
# ========= SETUP PACCHETTI =========
import sys, subprocess, importlib
def pip_install(pkg):
    try:
        importlib.import_module(pkg.split("==")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for pkg in ["openai>=1.51.0", "pandas", "openpyxl", "pypdf"]:
    pip_install(pkg)

# ========= IMPORT =========
import os, json, time
from pathlib import Path
import pandas as pd
from getpass import getpass
from openai import OpenAI

from pypdf import PdfReader, PdfWriter
import tempfile

MAX_PAGES = 8

def limit_pdf_pages(pdf_path: Path, max_pages: int = MAX_PAGES) -> Path:
    """
    Se il PDF ha più di max_pages pagine,
    crea una copia temporanea con solo le prime max_pages.
    Ritorna il path del PDF da usare (originale o ridotto).
    """
    reader = PdfReader(str(pdf_path))
    num_pages = len(reader.pages)

    if num_pages <= max_pages:
        return pdf_path  # usa originale

    writer = PdfWriter()
    for i in range(max_pages):
        writer.add_page(reader.pages[i])

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    with open(tmp.name, "wb") as f:
        writer.write(f)

    print(f"[INFO] {pdf_path.name}: {num_pages} pagine → tagliato a {max_pages}")
    return Path(tmp.name)


# ========= CONFIG BASE =========
# API key (Project key con permesso model.request)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("YourApi")

# Modello principale + fallback
MODEL_PRIMARY  = os.environ.get("OPENAI_MODEL", "gpt-5")
MODEL_FALLBACK = "gpt-5"

# Cartella con i 12 PDF (uno per mese)
INPUT_DIR = Path(".")        # <-- imposta la cartella corretta
PDF_PATTERN = "*.pdf"        # es: "*_2024.pdf" o "?? - ??_2024.pdf"

# File cumulativi da generare
OUT_CSV  = "estrazione_tutti_mesi.csv"
OUT_XLSX = "estrazione_tutti_mesi.xlsx"

# Parametri pipeline
RESUME   = False          # salta i file già processati se esiste OUT_CSV
RETRIES  = 3             # tentativi su errori temporanei
SLEEP_S  = 0.5           # pausa tra chiamate per rate limiting

# ========= PROMPT & SCHEMA =========
# ========= PROMPT & SCHEMA =========
PROMPT = """
OBIETTIVO
Estrarre da una bolletta luce/gas:
A) METADATI (1 per documento)
B) RIGHE DI DETTAGLIO: voci economiche MENSILI (ammesse righe negative)

PERIODO DI RIFERIMENTO (REGOLA FONDAMENTALE)
- Considera SOLO il livello MENSILE.
- IGNORA completamente dettagli giornalieri, orari o per fasce.
- Se coesistono dati giornalieri e mensili, usa SOLO quelli mensili.
- Estrai solo: periodo mensile, consumi totali del mese, corrispettivi mensili.

PRINCIPI GENERALI
- Le RIGHE contengono solo VOCI ECONOMICHE reali (corrispettivi, accise, conguagli, storni).
- NON inserire nelle righe: Totali, Imponibile, IVA, Totale documento/da pagare, Riepiloghi, Arrotondamenti.
- I valori di riepilogo vanno SOLO nei METADATI o in `imponibile_mese`.

⚠️ ECCEZIONE IMPOSTE
- ACCISE, ADDIZIONALI e IMPOSTE DI CONSUMO DEVONO SEMPRE essere incluse come RIGHE.
- Se esistono solo come totale aggregato → includi una sola riga con `manca_dettaglio="sì"`.

SEZIONE A) METADATI (se presenti)
- fornitore_nome, fornitore_piva
- numero_fattura, data_fattura (gg/mm/aaaa)
- periodo_inizio, periodo_fine (gg/mm/aaaa)
- cliente_nome, cliente_piva_cf
- indirizzo_fornitura
- pod (luce), pdr (gas)
- tensione_alimentazione
- potenza_disponibile, potenza_impegnata
- imponibile_mese
- totale_iva
- totale_documento / totale_da_pagare
- aliquote_iva_applicate

SEZIONE B) RIGHE DI DETTAGLIO (1 riga = 1 voce mensile)
- NON creare righe per giorni, fasce o date puntuali.
- Ogni riga rappresenta una VOCE ECONOMICA DI PERIODO.

Campi per riga:
- nome_cliente
- pod / pdr
- data_inizio, data_fine (eredita dal periodo se assenti)
- consumo_totale (obbligatorio nel mese; ripetilo su tutte le righe)
- dettaglio_voce (testo originale: es. materia energia, trasporto, accisa, UG1…)
- unita_misura
- quantita
- prezzo_aliquota
- importo
- imponibile_mese (uguale per tutte le righe del mese)
- tensione_alimentazione (se presente)
- potenza_disponibile / potenza_impegnata (se presenti)
- data_indirizzo (se presente)
- manca_dettaglio ("sì"/"no")
- note

REGOLE ANTI-DUPLICAZIONE
1) Escludi qualsiasi riga che sia un totale o riepilogo (Totale*, Imponibile*, IVA*, Riepilogo*, Arrotondamenti*).
2) Se una riga contiene la parola “Totale”, trattala sempre come totale → NON inserirla.
3) Usa solo il BLOCCO DI DETTAGLIO; usa riepilogo/sintesi SOLO se il dettaglio non esiste.
4) Categorie senza righe figlie (es. “Spesa per il trasporto…”, “Spesa per oneri…”, “Totale imposte”):
   - includi UNA riga sola
   - manca_dettaglio="sì"
   - NON includerle se esistono righe analitiche sottostanti.

IMPUTAZIONE IMPONIBILE
- `imponibile_mese` rappresenta il costo del periodo, indipendentemente da acconti/saldi.
- Se una sola voce vale 100 → imponibile_mese = 100.

NORMALIZZAZIONI
- Numeri: punto decimale, nessun separatore migliaia, mantieni segno.
- Date: gg/mm/aaaa.
- Testi: usa esattamente le diciture della bolletta.

OUTPUT
- Restituisci SOLO JSON:
{
  "metadati": { ... },
  "rows": [ ... ]
}
- Nessun testo fuori dal JSON.

"""

JSON_SCHEMA = {
    "name": "righe_bolletta",
    "schema": {
        "type": "object",
        "properties": {
            "rows": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "nome_cliente": {"type": "string"},
                        "pdr": {"type": "string"},
                        "pod": {"type": "string"},
                        "data_inizio": {"type": "string"},
                        "data_fine": {"type": "string"},
                        "consumo_totale": {"type": ["number", "string"]},
                        "dettaglio_voce": {"type": "string"},
                        "unita_misura": {"type": "string"},
                        "quantita": {"type": ["number", "string"]},
                        "prezzo_aliquota": {"type": ["number", "string"]},
                        "importo": {"type": ["number", "string"]},
                        "imponibile_mese": {"type": ["number", "string"]},
                        "note": {"type": "string"},
                        "tensione_alimentazione": {"type": "string"},
                        "manca_dettaglio": {"type": "string"},
                        "potenza_disponibile": {"type": "string"},
                        "potenza_impegnata": {"type": "string"},
                        "data_indirizzo": {"type": "string"}
                    },
                    "required": [
                        "nome_cliente",
                        "pdr",
                        "data_inizio",
                        "data_fine",
                        "consumo_totale",
                        "dettaglio_voce",
                        "importo",
                        "imponibile_mese"
                        "manca_dettaglio"
                    ]
                }
            }
        },
        "required": ["rows"],
        "additionalProperties": False
    },
    "strict": True
}

# ========= CLIENT =========
client = OpenAI()
#client = OpenAI(timeout=120)


# ========= FUNZIONI CORE =========
def extract_rows_from_pdf(pdf_path: Path, model: str) -> list[dict]:
    if not pdf_path.exists():
        raise FileNotFoundError(f"File non trovato: {pdf_path}")

    # 🔥 TAGLIO HARD A MAX 8 PAGINE
    pdf_to_use = limit_pdf_pages(pdf_path, MAX_PAGES)

    with open(pdf_to_use, "rb") as f:
        uploaded = client.files.create(file=f, purpose="user_data")


    # Tentativo A: Structured Outputs con response_format
    try:
        rsp = client.responses.create(
            model=model,
            instructions="Analizza il PDF e restituisci SOLO JSON aderente allo schema.",
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": PROMPT},
                    {"type": "input_file", "file_id": uploaded.id}
                ]
            }],
            response_format={"type": "json_schema", "json_schema": JSON_SCHEMA}, #da attivare
            max_output_tokens=200_000
        )
        raw = getattr(rsp, "output_text", None)
        if not raw:
            parts = []
            for out in getattr(rsp, "output", []) or []:
                for c in getattr(out, "content", []) or []:
                    if getattr(c, "type", "") == "output_text":
                        parts.append(c.text)
            raw = "".join(parts)
    except TypeError:
        # Tentativo B: SDK senza response_format -> vincolo via prompt
        rsp = client.responses.create(
            model=model,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text":
                        f"{PROMPT}\n\nRESTITUISCI SOLO JSON valido con radice {{\"rows\": [...]}} "
                        f"aderente ESATTAMENTE a questo schema (nessun testo extra):\n{json.dumps(JSON_SCHEMA)}"
                    },
                    {"type": "input_file", "file_id": uploaded.id}
                ]
            }],
            max_output_tokens=200_000
        )
        raw = getattr(rsp, "output_text", "") or ""
        if not raw:
            parts = []
            for out in getattr(rsp, "output", []) or []:
                for c in getattr(out, "content", []) or []:
                    if getattr(c, "type", "") == "output_text":
                        parts.append(c.text)
            raw = "".join(parts)
        s, e = raw.find("{"), raw.rfind("}")
        if s >= 0 and e > s:
            raw = raw[s:e+1]

    if not raw:
        raise RuntimeError(f"Risposta vuota dal modello su: {pdf_path.name}")

    data = json.loads(raw)  # alza se non JSON valido
    rows = data.get("rows", [])
    for r in rows:
        r["_source_file"] = pdf_path.name
    return rows


def process_all_pdfs(input_dir: Path,
                     pattern: str,
                     out_csv: str,
                     out_xlsx: str,
                     resume: bool = True,
                     retries: int = 3,
                     sleep_s: float = 0.5) -> pd.DataFrame | None:
    """Elabora tutti i PDF (uno alla volta), accoda, salva CSV/XLSX unici."""
    pdfs = sorted(input_dir.glob(pattern))
    if not pdfs:
        print(f"Nessun PDF trovato in {input_dir} con pattern {pattern}")
        return None

    all_rows, processed = [], set()
    if resume and Path(out_csv).exists():
        prev = pd.read_csv(out_csv, dtype=str)
        if "_source_file" in prev.columns:
            processed = set(prev["_source_file"].dropna().unique().tolist())
        all_rows.extend(prev.to_dict(orient="records"))
        print(f"[RESUME] righe esistenti: {len(all_rows)} | file già fatti: {len(processed)}")

    for fp in pdfs:
        if fp.name in processed:
            print(f"[SKIP] {fp.name}")
            continue

        print(f"[PROCESS] {fp.name}")
        err = None
        for attempt in range(1, retries + 1):
            try:
                try:
                    rows = extract_rows_from_pdf(fp, MODEL_PRIMARY)
                except Exception as e1:
                    print(f"  [WARN] {MODEL_PRIMARY} errore: {e1} → provo {MODEL_FALLBACK}")
                    rows = extract_rows_from_pdf(fp, MODEL_FALLBACK)
                all_rows.extend(rows)
                err = None
                break
            except Exception as e:
                err = e
                wait = min(2 ** attempt, 15)
                print(f"  tentativo {attempt}/{retries} fallito: {e} → retry in {wait}s")
                time.sleep(wait)

        if err:
            print(f"[ERROR] {fp.name}: {err}")
            continue

        # salvataggio incrementale
        pd.DataFrame(all_rows).to_csv(out_csv, index=False, encoding="utf-8")
        print(f"  +{len(rows)} righe (totale: {len(all_rows)})")
        if sleep_s > 0:
            time.sleep(sleep_s)

    if not all_rows:
        print("[DONE] nessuna riga estratta.")
        return None

    df = pd.DataFrame(all_rows)
    df.to_excel(out_xlsx, index=False)
    print(f"[DONE] Righe totali: {len(df)}")
    print(f"  CSV : {out_csv}")
    print(f"  XLSX: {out_xlsx}")
    return df


# ========= FUNZIONE OPZIONALE: UNIRE EXCEL PER-FILE =========
def merge_excels_in_folder(input_dir: Path,
                           pattern: str = "*.xlsx",
                           out_xlsx: str = "merge_excel_finale.xlsx") -> pd.DataFrame | None:
    """Se hai già creato tanti Excel singoli, li unisce in un unico file."""
    xls_files = sorted(input_dir.glob(pattern))
    if not xls_files:
        print(f"Nessun XLSX trovato in {input_dir} (pattern: {pattern})")
        return None

    frames = []
    for xf in xls_files:
        try:
            df = pd.read_excel(xf, dtype=str)
            df["_source_excel"] = xf.name
            frames.append(df)
            print(f"[OK] unito: {xf.name} ({len(df)} righe)")
        except Exception as e:
            print(f"[WARN] salto {xf.name}: {e}")

    if not frames:
        print("Nessun Excel valido da unire.")
        return None

    big = pd.concat(frames, ignore_index=True)
    big.to_excel(out_xlsx, index=False)
    print(f"[DONE] Excel unificato: {out_xlsx} (righe: {len(big)})")
    return big


# ========= ESECUZIONE =========
# 1) ELABORA TUTTI I PDF UNO ALLA VOLTA → UNICO CSV/XLSX
df_all = process_all_pdfs(
    input_dir=INPUT_DIR,
    pattern=PDF_PATTERN,
    out_csv=OUT_CSV,
    out_xlsx=OUT_XLSX,
    resume=RESUME,
    retries=RETRIES,
    sleep_s=SLEEP_S,
)

# 2) (OPZIONALE) UNISCI EXCEL GIÀ ESISTENTI “PER-FILE”
# merge_excels_in_folder(INPUT_DIR, pattern="*.perfile.xlsx", out_xlsx="merge_excel_finale.xlsx")

# 3) Vedi anteprima nel notebook
if isinstance(df_all, pd.DataFrame):
    display(df_all.head(30))


In [ ]:
"""# ========= SETUP PACCHETTI =========
import sys, subprocess, importlib
def pip_install(pkg):
    try:
        importlib.import_module(pkg.split("==")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for pkg in ["openai>=1.51.0", "pandas", "openpyxl", "pypdf"]:
    pip_install(pkg)

# ========= IMPORT =========
import os, json, time
from pathlib import Path
import pandas as pd
from getpass import getpass
from openai import OpenAI

from pypdf import PdfReader, PdfWriter
import tempfile

MAX_PAGES = 8

def limit_pdf_pages(pdf_path: Path, max_pages: int = MAX_PAGES) -> Path:
    """
    Se il PDF ha più di max_pages pagine,
    crea una copia temporanea con solo le prime max_pages.
    Ritorna il path del PDF da usare (originale o ridotto).
    """
    reader = PdfReader(str(pdf_path))
    num_pages = len(reader.pages)

    if num_pages <= max_pages:
        return pdf_path  # usa originale

    writer = PdfWriter()
    for i in range(max_pages):
        writer.add_page(reader.pages[i])

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pdf")
    with open(tmp.name, "wb") as f:
        writer.write(f)

    print(f"[INFO] {pdf_path.name}: {num_pages} pagine → tagliato a {max_pages}")
    return Path(tmp.name)


# ========= CONFIG BASE =========
# API key (Project key con permesso model.request)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Your_Api")

# Modello principale + fallback
MODEL_PRIMARY  = os.environ.get("OPENAI_MODEL", "gpt-5")
MODEL_FALLBACK = "gpt-5"

# Cartella con i 12 PDF (uno per mese)
INPUT_DIR = Path(".")        # <-- imposta la cartella corretta
PDF_PATTERN = "*.pdf"        # es: "*_2024.pdf" o "?? - ??_2024.pdf"

# File cumulativi da generare
OUT_CSV  = "estrazione_tutti_mesi.csv"
OUT_XLSX = "estrazione_tutti_mesi.xlsx"

# Parametri pipeline
RESUME   = False          # salta i file già processati se esiste OUT_CSV
RETRIES  = 3             # tentativi su errori temporanei
SLEEP_S  = 0.5           # pausa tra chiamate per rate limiting

# ========= PROMPT & SCHEMA =========
# ========= PROMPT & SCHEMA =========
PROMPT = """
OBIETTIVO
Estrarre da ogni bolletta (luce/gas):
A) METADATI/TESTATA (un solo blocco per documento)
B) RIGHE DI DETTAGLIO dei corrispettivi (tabella; ammesse righe negative)

REGOLE SUL PERIODO DI RIFERIMENTO
- IGNORA completamente eventuali dettagli giornalieri, orari o per singolo giorno.
- Se presenti tabelle di consumo GIORNALIERE o per fasce giornaliere, NON estrarle.
- Valuta ed estrai ESCLUSIVAMENTE:
  - il PERIODO MENSILE complessivo
  - i CONSUMI TOTALI del mese
  - i CORRISPETTIVI MENSILI aggregati
- Se coesistono dati giornalieri e mensili, considera SOLO il livello mensile.

PRINCIPI CHIAVE
- Le RIGHE DI DETTAGLIO devono contenere solo voci economiche effettive (anche “-”: Acconti, Storni, Conguagli, Ricalcoli, ecc.).
- Nessuna voce di riepilogo/totale nelle RIGHE (es.: Imponibile, IVA, Totale documento/da pagare, Riepilogo, Arrotondamenti).
- I valori di riepilogo vanno solo nei METADATI (oppure in una colonna non di dettaglio come `imponibile_mese`).

SEZIONE A) METADATI / TESTATA (1 per documento)
Compila, se presenti:
- fornitore_nome
- fornitore_piva
- numero_fattura
- data_fattura (gg/mm/aaaa)
- periodo_inizio (gg/mm/aaaa) – periodo di competenza
- periodo_fine (gg/mm/aaaa)
- cliente_nome (ragione sociale)
- cliente_piva_cf
- indirizzo_fornitura
- pod (energia elettrica)
- pdr (gas)
- tensione_alimentazione
- potenza_disponibile
- potenza_impegnata
- imponibile_mese (imponibile del periodo; unico riepilogo)
- totale_iva
- Potenza disponibile (se presente)
- Potenza impegnata (se presente)
- totale_documento (o “totale da pagare”)
- aliquote_iva_applicate (elenco/testo)

SEZIONE B) RIGHE DI DETTAGLIO (tabella; una riga per voce economica)
- NON creare righe separate per singoli giorni, fasce orarie o date puntuali.
- Ogni riga deve rappresentare una VOCE ECONOMICA DI PERIODO MENSILE.
- Non inserire la riga dell'iva nel dettaglio, altrimenti poi se sommassi tutti i dettagli con la stessa data non verrebbe corretto l'imponibile
Per ogni riga estrai:
- nome_cliente
- pdr (se gas)
- pod (se elettrico)
- data_inizio (gg/mm/aaaa) – inizio competenza riga
- data_fine (gg/mm/aaaa) – fine competenza riga
- consumo_totale (obbligatorio in quel mese)
- dettaglio_voce (es.: “Quota variabile materia prima gas”, “Componente Pfix”, “UG1”, “CRVOS”, “CVu trasporto”, ecc.)
- unita_misura (es.: Smc, kWh, N.)
- quantita (numero; può essere negativo)
- prezzo_aliquota (numero decimale)
- importo (numero decimale; può essere negativo)
- imponibile_mese (valore di periodo; non di dettaglio; può ripetersi identico su tutte le righe del mese)
- note
- tensione_alimentazione (se presente)
- potenza_disponibile (se presente)
- potenza_impegnata (se presente)
- data_indirizzo (se presente)
- manca dettaglio

REGOLE ANTI-DUPLICAZIONE (case-insensitive; anche varianti)
1) ESCLUDI dalle righe tutte le voci di riepilogo/totali: “Totale”, “Totali”, “Totale fattura”, “Totale documento”, “Totale da pagare”,
   “Imponibile”, “Imponibile totale”, “Riepilogo”, “Riepilogo imponibile”, “IVA”, “Totale IVA”, “Arrotondamenti”.
   Questi valori vanno SOLO nei METADATI (o in `imponibile_mese`), NON come riga.
2) Se una riga ha un’etichetta di dettaglio ma include “Totale …”, TRATTALA come totale e NON inserirla tra i dettagli.
3) Prendi solo importi/righe all’interno del blocco di dettaglio. Se non esiste un blocco di dettaglio, solo in quel caso valuta di leggere il blocco "riepilogo" o "sintesi" che sia, altrimenti basati solo sul dettaglio ignorando il resto..
3-bis) Eccezione “totale di sezione senza dettaglio”:
Se in tabella compare una riga di categoria (es. “SPESA PER LA MATERIA GAS…”, 
“SPESA PER IL TRASPORTO E LA GESTIONE DEL CONTATORE”, “SPESA PER ONERI DI SISTEMA”, 
“ALTRE PARTITE”, “TOTALE IMPOSTE”) e NON sono presenti righe figlie/analitiche per quella categoria
nello stesso documento/periodo, allora quella riga di categoria va inclusa come RIGA DI DETTAGLIO
(una riga sola), con:
- dettaglio_voce = il testo della categoria (es. “Spesa per il trasporto e la gestione del contatore”)
- importo = l’importo indicato
- unita_misura, prezzo_aliquota, quantita = vuoti se non esplicitati
- manca_dettaglio = "sì"
NON includerla se esistono righe analitiche sottostanti: in quel caso resta un totale da escludere.

3-ter) “TOTALE IMPOSTE”:
- Se sono presenti righe analitiche di imposte (accisa, addizionale, …) → escludilo.
- Se NON esistono righe analitiche di imposte → includilo come sopra (manca_dettaglio="sì").

4) IMPUTAZIONE DELL’IMPONIBILE: considera l’importo dovuto per quel periodo a prescindere da acconti/saldi pregressi. Se i dettagli sono 0 e una sola voce vale 100, allora `imponibile_mese = 100`.
5) metti lo stesso campo consumo gas in tutte le righe di quel mese. E' un campo sempre presente
6) quando il dettaglio non è completo (ovvero è presente solo accisa, imposta erariale ma senza tutti gli altri costi) allora metti nella colonna "manca dettaglio" il valore "sì"
NORMALIZZAZIONI
- Numeri: punto come separatore decimale (es. 1234.56), nessun separatore migliaia; preserva il segno “-”.
- Date: formato gg/mm/aaaa. Se la riga non ha date, eredita `periodo_inizio` e `periodo_fine` dai METADATI.
- Testi: mantieni esattamente le diciture delle voci come in bolletta.
OUTPUT (NESSUN TESTO EXTRA)
- Restituisci UN SOLO oggetto JSON con due chiavi di primo livello:
  {
    "metadati": { ... },
    "rows": [ ... ]
  }
- Se non esistono righe valide, restituisci `"rows": []` ma compila i METADATI se disponibili.
- Non aggiungere commenti, spiegazioni o testo fuori dal JSON.
"""

JSON_SCHEMA = {
    "name": "righe_bolletta",
    "schema": {
        "type": "object",
        "properties": {
            "rows": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "nome_cliente": {"type": "string"},
                        "pdr": {"type": "string"},
                        "pod": {"type": "string"},
                        "data_inizio": {"type": "string"},
                        "data_fine": {"type": "string"},
                        "consumo_totale": {"type": ["number", "string"]},
                        "dettaglio_voce": {"type": "string"},
                        "unita_misura": {"type": "string"},
                        "quantita": {"type": ["number", "string"]},
                        "prezzo_aliquota": {"type": ["number", "string"]},
                        "importo": {"type": ["number", "string"]},
                        "imponibile_mese": {"type": ["number", "string"]},
                        "note": {"type": "string"},
                        "tensione_alimentazione": {"type": "string"},
                        "manca_dettaglio": {"type": "string"},
                        "potenza_disponibile": {"type": "string"},
                        "potenza_impegnata": {"type": "string"},
                        "data_indirizzo": {"type": "string"}
                    },
                    "required": [
                        "nome_cliente",
                        "pdr",
                        "data_inizio",
                        "data_fine",
                        "consumo_totale",
                        "dettaglio_voce",
                        "importo",
                        "imponibile_mese"
                        "manca_dettaglio"
                    ]
                }
            }
        },
        "required": ["rows"],
        "additionalProperties": False
    },
    "strict": True
}

# ========= CLIENT =========
client = OpenAI()
#client = OpenAI(timeout=120)


# ========= FUNZIONI CORE =========
def extract_rows_from_pdf(pdf_path: Path, model: str) -> list[dict]:
    if not pdf_path.exists():
        raise FileNotFoundError(f"File non trovato: {pdf_path}")

    # 🔥 TAGLIO HARD A MAX 8 PAGINE
    pdf_to_use = limit_pdf_pages(pdf_path, MAX_PAGES)

    with open(pdf_to_use, "rb") as f:
        uploaded = client.files.create(file=f, purpose="user_data")


    # Tentativo A: Structured Outputs con response_format
    try:
        rsp = client.responses.create(
            model=model,
            instructions="Analizza il PDF e restituisci SOLO JSON aderente allo schema.",
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": PROMPT},
                    {"type": "input_file", "file_id": uploaded.id}
                ]
            }],
            response_format={"type": "json_schema", "json_schema": JSON_SCHEMA}, #da attivare
            max_output_tokens=200_000
        )
        raw = getattr(rsp, "output_text", None)
        if not raw:
            parts = []
            for out in getattr(rsp, "output", []) or []:
                for c in getattr(out, "content", []) or []:
                    if getattr(c, "type", "") == "output_text":
                        parts.append(c.text)
            raw = "".join(parts)
    except TypeError:
        # Tentativo B: SDK senza response_format -> vincolo via prompt
        rsp = client.responses.create(
            model=model,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text":
                        f"{PROMPT}\n\nRESTITUISCI SOLO JSON valido con radice {{\"rows\": [...]}} "
                        f"aderente ESATTAMENTE a questo schema (nessun testo extra):\n{json.dumps(JSON_SCHEMA)}"
                    },
                    {"type": "input_file", "file_id": uploaded.id}
                ]
            }],
            max_output_tokens=200_000
        )
        raw = getattr(rsp, "output_text", "") or ""
        if not raw:
            parts = []
            for out in getattr(rsp, "output", []) or []:
                for c in getattr(out, "content", []) or []:
                    if getattr(c, "type", "") == "output_text":
                        parts.append(c.text)
            raw = "".join(parts)
        s, e = raw.find("{"), raw.rfind("}")
        if s >= 0 and e > s:
            raw = raw[s:e+1]

    if not raw:
        raise RuntimeError(f"Risposta vuota dal modello su: {pdf_path.name}")

    data = json.loads(raw)  # alza se non JSON valido
    rows = data.get("rows", [])
    for r in rows:
        r["_source_file"] = pdf_path.name
    return rows


def process_all_pdfs(input_dir: Path,
                     pattern: str,
                     out_csv: str,
                     out_xlsx: str,
                     resume: bool = True,
                     retries: int = 3,
                     sleep_s: float = 0.5) -> pd.DataFrame | None:
    """Elabora tutti i PDF (uno alla volta), accoda, salva CSV/XLSX unici."""
    pdfs = sorted(input_dir.glob(pattern))
    if not pdfs:
        print(f"Nessun PDF trovato in {input_dir} con pattern {pattern}")
        return None

    all_rows, processed = [], set()
    if resume and Path(out_csv).exists():
        prev = pd.read_csv(out_csv, dtype=str)
        if "_source_file" in prev.columns:
            processed = set(prev["_source_file"].dropna().unique().tolist())
        all_rows.extend(prev.to_dict(orient="records"))
        print(f"[RESUME] righe esistenti: {len(all_rows)} | file già fatti: {len(processed)}")

    for fp in pdfs:
        if fp.name in processed:
            print(f"[SKIP] {fp.name}")
            continue

        print(f"[PROCESS] {fp.name}")
        err = None
        for attempt in range(1, retries + 1):
            try:
                try:
                    rows = extract_rows_from_pdf(fp, MODEL_PRIMARY)
                except Exception as e1:
                    print(f"  [WARN] {MODEL_PRIMARY} errore: {e1} → provo {MODEL_FALLBACK}")
                    rows = extract_rows_from_pdf(fp, MODEL_FALLBACK)
                all_rows.extend(rows)
                err = None
                break
            except Exception as e:
                err = e
                wait = min(2 ** attempt, 15)
                print(f"  tentativo {attempt}/{retries} fallito: {e} → retry in {wait}s")
                time.sleep(wait)

        if err:
            print(f"[ERROR] {fp.name}: {err}")
            continue

        # salvataggio incrementale
        pd.DataFrame(all_rows).to_csv(out_csv, index=False, encoding="utf-8")
        print(f"  +{len(rows)} righe (totale: {len(all_rows)})")
        if sleep_s > 0:
            time.sleep(sleep_s)

    if not all_rows:
        print("[DONE] nessuna riga estratta.")
        return None

    df = pd.DataFrame(all_rows)
    df.to_excel(out_xlsx, index=False)
    print(f"[DONE] Righe totali: {len(df)}")
    print(f"  CSV : {out_csv}")
    print(f"  XLSX: {out_xlsx}")
    return df


# ========= FUNZIONE OPZIONALE: UNIRE EXCEL PER-FILE =========
def merge_excels_in_folder(input_dir: Path,
                           pattern: str = "*.xlsx",
                           out_xlsx: str = "merge_excel_finale.xlsx") -> pd.DataFrame | None:
    """Se hai già creato tanti Excel singoli, li unisce in un unico file."""
    xls_files = sorted(input_dir.glob(pattern))
    if not xls_files:
        print(f"Nessun XLSX trovato in {input_dir} (pattern: {pattern})")
        return None

    frames = []
    for xf in xls_files:
        try:
            df = pd.read_excel(xf, dtype=str)
            df["_source_excel"] = xf.name
            frames.append(df)
            print(f"[OK] unito: {xf.name} ({len(df)} righe)")
        except Exception as e:
            print(f"[WARN] salto {xf.name}: {e}")

    if not frames:
        print("Nessun Excel valido da unire.")
        return None

    big = pd.concat(frames, ignore_index=True)
    big.to_excel(out_xlsx, index=False)
    print(f"[DONE] Excel unificato: {out_xlsx} (righe: {len(big)})")
    return big


# ========= ESECUZIONE =========
# 1) ELABORA TUTTI I PDF UNO ALLA VOLTA → UNICO CSV/XLSX
df_all = process_all_pdfs(
    input_dir=INPUT_DIR,
    pattern=PDF_PATTERN,
    out_csv=OUT_CSV,
    out_xlsx=OUT_XLSX,
    resume=RESUME,
    retries=RETRIES,
    sleep_s=SLEEP_S,
)

# 2) (OPZIONALE) UNISCI EXCEL GIÀ ESISTENTI “PER-FILE”
# merge_excels_in_folder(INPUT_DIR, pattern="*.perfile.xlsx", out_xlsx="merge_excel_finale.xlsx")

# 3) Vedi anteprima nel notebook
if isinstance(df_all, pd.DataFrame):
    display(df_all.head(30))
"""
script vecchio funzionante ma che spesso andava in timeout, è stato quindi sostituito da un prompt più snello

In [ ]:
import pandas as pd
import numpy as np
import csv
import re
from pathlib import Path

# ========= CONFIG =========
INPUT_PATH = "estrazione_tutti_mesi.csv"  # <-- cambia se serve
OUT_XLSX   = "bollette_raggruppate.xlsx"                    # facoltativo

# ========= DISPLAY (per debug) =========
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 2000)
pd.set_option("display.colheader_justify", "left")

# ========= UTILS =========
def clean_colname(s: str) -> str:
    if s is None:
        return ""
    s = re.sub(r"[\uFEFF\u200B\u200C\u200D]", "", str(s))  # BOM/zero-width
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def sniff_sep(sample_path: str) -> str:
    with open(sample_path, "r", encoding="utf-8", errors="ignore") as f:
        sample = f.read(4096)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[",",";","\t","|"])
        return dialect.delimiter
    except Exception:
        return "," if sample.count(",") >= sample.count(";") else ";"

def read_table(path: str) -> pd.DataFrame:
    p = Path(path)
    if p.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(p, dtype=str)
    else:
        sep = sniff_sep(path)
        print(f"🔎 Separatore rilevato: {repr(sep)}")
        df = pd.read_csv(p, sep=sep, engine="python", dtype=str)
    return df.rename(columns=lambda c: clean_colname(c))

def find_col(df: pd.DataFrame, candidates) -> str | None:
    cols = list(df.columns)
    for cand in candidates:
        c = clean_colname(cand)
        if c in cols:
            return c
    # match "soft": inizio o contiene
    for cand in candidates:
        c = clean_colname(cand)
        for col in cols:
            if col.startswith(c) or c in col:
                return col
    return None

def to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    # se presenti sia '.' che ',' -> '.' migliaia, ',' decimale
    if "." in s and "," in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(",", ".")
    s = re.sub(r"[^0-9\.\-]", "", s)  # rimuovi simboli non numerici
    if s in ("", "-", ".", "-."):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

def month_name_it(m: int) -> str:
    mesi = {
        1:"gennaio",2:"febbraio",3:"marzo",4:"aprile",5:"maggio",6:"giugno",
        7:"luglio",8:"agosto",9:"settembre",10:"ottobre",11:"novembre",12:"dicembre"
    }
    return mesi.get(int(m), "")

def consumo_moda(series: pd.Series) -> float:
    """
    NON sommare: consumo del mese come MODA.
    - Se unica moda -> quella
    - Se più mode -> prendi la prima (equivalente al valore più frequente)
    - Se non ci sono numeri -> NaN
    - Fallback: ultimo non-NaN
    """
    s = series.dropna().astype(float).round(6)
    if s.empty:
        return np.nan
    mode = s.mode()
    if len(mode) >= 1:
        return float(mode.iloc[0])
    return float(s.iloc[-1])

def is_si(x: str) -> bool:
    """Ritorna True se il testo equivale a 'sì'/'si' (case-insensitive, spazi ignorati)."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return False
    t = str(x).strip().lower()
    t = t.replace("ì","i")  # gestisci accento mancante
    return t in {"si", "sì", "yes", "true", "1", "y"}

# ========= 1) CARICA =========
df = read_table(INPUT_PATH)

# ========= 2) MAPPATURA COLONNE =========
col_importo        = find_col(df, ["importo","importi","valore","totale_riga"])
col_imponibile     = find_col(df, ["imponibile_mese","imponibile mese","imponibile"])
col_cons           = find_col(df, ["consumo_totale","consumo","consumi"])
col_manca_dett     = find_col(df, ["manca_dettaglio","manca dettaglio","senza_dettaglio","senza dettaglio"])
col_voce           = find_col(df, ["dettaglio_voce","voce","descrizione","descrizione_voce"])
# per competenza temporale meglio 'data_fine' -> 'data_inizio' -> 'data_fattura' -> 'data'
col_data = (find_col(df, ["data_fine"])
            or find_col(df, ["data_inizio"])
            or find_col(df, ["data_fattura"])
            or find_col(df, ["data"]))

missing = []
if not col_cons:       missing.append("consumo_totale/consumo")
if not col_data:       missing.append("data_fine/data_inizio/data_fattura/data")
# almeno uno tra importo o imponibile deve esserci
if not col_importo and not col_imponibile:
    missing.append("importo o imponibile_mese")
if missing:
    raise ValueError("Colonne mancanti nel file: " + ", ".join(missing))

print("✅ Colonne usate ->",
      f"consumo: '{col_cons}', data: '{col_data}',",
      f"importo: '{col_importo}'" if col_importo else "importo: (assente)",
      f"imponibile_mese: '{col_imponibile}'" if col_imponibile else "imponibile_mese: (assente)",
      f"manca_dettaglio: '{col_manca_dett}'" if col_manca_dett else "manca_dettaglio: (assente)",
      f"voce: '{col_voce}'" if col_voce else "voce: (assente)")

# ========= 3) CONVERSIONI =========
if col_importo:
    df[col_importo] = df[col_importo].apply(to_float)
if col_imponibile:
    df[col_imponibile] = df[col_imponibile].apply(to_float)
df[col_cons] = df[col_cons].apply(to_float)
df[col_data] = pd.to_datetime(df[col_data], dayfirst=True, errors="coerce")

# Normalizza 'manca_dettaglio'
if col_manca_dett:
    df["_manca_det_bool"] = df[col_manca_dett].apply(is_si)
else:
    df["_manca_det_bool"] = False

# Scarta righe senza data valida
prima = len(df)
df = df.dropna(subset=[col_data]).copy()
dopo  = len(df)
if dopo < prima:
    print(f"⚠️ Righe senza data valida scartate: {prima - dopo}")

# ========= 4) CHIAVI TEMPORALI =========
df["anno"] = df[col_data].dt.year
df["mese_num"] = df[col_data].dt.month
df["mese"] = df["mese_num"].apply(month_name_it)

# ========= DEBUG: PRIMA DEL RAGGRUPPAMENTO =========
cols_debug = [c for c in [col_data, "anno", "mese_num", "mese",
                          col_cons, col_importo, col_imponibile,
                          col_voce, col_manca_dett, "_manca_det_bool"] if c in df.columns or c=="_manca_det_bool"]
print("\n📄 DATAFRAME PRIMA DEL RAGGRUPPAMENTO (prime 120 righe):")
print(df[cols_debug].head(120).to_string(index=False))

# ========= 5) RAGGRUPPAMENTO CON LOGICA “MANCA DETTAGLIO” =========
def agg_per_mese(grp: pd.DataFrame) -> pd.Series:
    # Consumo del mese: MODA (non sommare)
    consumo = consumo_moda(grp[col_cons])

    # Flag "manca dettaglio" per il mese (se QUALSIASI riga del mese dice sì, trattiamo come manca dettaglio)
    manca_det_mese = bool(grp["_manca_det_bool"].any())

    if manca_det_mese:
        # IMPORTI: NON sommare; usa imponibile_mese (uno a caso tra i non NaN)
        if col_imponibile and col_imponibile in grp.columns:
            imp_vals = grp[col_imponibile].dropna()
            importo_finale = float(imp_vals.iloc[0]) if not imp_vals.empty else np.nan
        else:
            # fallback estremo: se manca proprio la colonna imponibile_mese,
            # non sommiamo comunque; prendiamo l'ultimo importo non-NaN
            if col_importo and col_importo in grp.columns:
                imp_vals = grp[col_importo].dropna()
                importo_finale = float(imp_vals.iloc[-1]) if not imp_vals.empty else np.nan
            else:
                importo_finale = np.nan
    else:
        # Dettaglio presente: importi = somma delle righe
        if col_importo and col_importo in grp.columns:
            importo_finale = float(grp[col_importo].sum())
        else:
            # se manca 'importo', prova a usare comunque l'imponibile_mese (prendi il primo)
            if col_imponibile and col_imponibile in grp.columns:
                imp_vals = grp[col_imponibile].dropna()
                importo_finale = float(imp_vals.iloc[0]) if not imp_vals.empty else np.nan
            else:
                importo_finale = np.nan

    # info diagnostiche
    consumo_nuniq = grp[col_cons].dropna().astype(float).round(6).nunique()
    imponibile_nuniq = grp[col_imponibile].dropna().nunique() if col_imponibile and col_imponibile in grp.columns else np.nan

    return pd.Series({
        "totale_importi": importo_finale,
        "consumo_mese": consumo,
        "manca_dettaglio_mese": manca_det_mese,
        "consumo_valori_distinti": consumo_nuniq,
        "imponibile_valori_distinti": imponibile_nuniq,
        "righe": len(grp)
    })

agg = df.groupby(["anno","mese_num"], as_index=False).apply(agg_per_mese)

# Inserisci nome mese e ordina
agg["mese"] = agg["mese_num"].apply(month_name_it)
agg = agg.sort_values(["anno", "mese_num"]).reset_index(drop=True)

# ========= DEBUG: DOPO IL RAGGRUPPAMENTO =========
print("\n📊 RIEPILOGO PER MESE (Totale Importi + Consumo del mese):")
print(agg[["anno","mese_num","mese","totale_importi","consumo_mese",
           "manca_dettaglio_mese","consumo_valori_distinti","imponibile_valori_distinti","righe"]]
      .to_string(index=False))

# ========= (FACOLTATIVO) ESPORTA =========
#Nota: se vuoi i numeri con virgola in Excel (ma restano testo):
out = agg[["anno","mese","totale_importi","consumo_mese"]].copy()
out["totale_importi"] = out["totale_importi"].map(lambda v: ("" if pd.isna(v) else f"{v:.2f}".replace(".", ",")))
out["consumo_mese"]   = out["consumo_mese"].map(lambda v: ("" if pd.isna(v) else str(v).replace(".", ",")))
out.to_excel(OUT_XLSX, index=False)
print(f"\n✅ Esportato: {OUT_XLSX}")
